In [1]:
# ============================================================
# PS S6E5 - exp_20260504_010_blend_core_min
#
# Compare:
# - Avg
# - Ridge
# - ElasticNet - disabled
# - LogisticRegression
# - Hill Climb
# - Nelder-Mead
# - Signed SLSQP
# ============================================================

In [2]:
import os
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.optimize import minimize
from scipy.stats import rankdata, spearmanr

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, ElasticNet, LogisticRegression

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

In [3]:
# ============================================================
# Config
# ============================================================

class CFG:
    EXP_ID = "exp_20260504_010_blend_core_min"
    COMPETITION = "PS S6E5 Predicting F1 Pit Stops"
    METRIC = "AUC"

    COMP_BASE = Path("/kaggle/input/competitions/playground-series-s6e5")
    TRAIN_PATH = COMP_BASE / "train.csv"
    SUB_PATH = COMP_BASE / "sample_submission.csv"

    NPY_BASE = Path("/kaggle/input/datasets/mizushimatoshihiko/npy-files")

    OUTDIR = Path(f"/kaggle/working/{EXP_ID}")

    ID_COL = "id"
    TARGET = "PitNextLap"

    SEED = 42
    N_SPLITS = 5

    EPS = 1e-6


CFG.OUTDIR.mkdir(parents=True, exist_ok=True)

In [4]:
# ============================================================
# Utilities
# ============================================================

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def safe_clip(x, eps=CFG.EPS):
    return np.clip(np.asarray(x, dtype=float), eps, 1.0 - eps)


def safe_logit(p, eps=CFG.EPS):
    p = safe_clip(p, eps)
    return np.log(p / (1.0 - p))


def rank01(x):
    x = np.asarray(x, dtype=float)
    return rankdata(x, method="average") / (len(x) + 1.0)


def softmax(z):
    z = np.asarray(z, dtype=float)
    z = z - np.max(z)
    e = np.exp(z)
    return e / e.sum()


def normalize_weights(w):
    w = np.asarray(w, dtype=float)
    w = np.clip(w, 0.0, None)
    s = w.sum()
    if s <= 0:
        return np.ones_like(w) / len(w)
    return w / s


def weighted_average(X, w):
    w = normalize_weights(w)
    return np.asarray(X) @ w


def auc(y, pred):
    return roc_auc_score(y, pred)


def sanitize_name(name: str) -> str:
    return (
        name.replace(" ", "_")
            .replace("/", "_")
            .replace("+", "plus")
            .replace("-", "_")
            .replace(".", "_")
    )


seed_everything(CFG.SEED)

In [5]:
# ============================================================
# Load data
# ============================================================

train = pd.read_csv(CFG.TRAIN_PATH)
sub = pd.read_csv(CFG.SUB_PATH)

y = train[CFG.TARGET].astype(int).values

print("train:", train.shape)
print("sub  :", sub.shape)
print("target mean:", y.mean())

train: (439140, 16)
sub  : (188165, 2)
target mean: 0.19898210137996994


In [6]:
# ============================================================
# Load OOF / pred
# ============================================================

MODEL_SPECS = [
    # {
    #     "key": "007",
    #     "name": "007_realmlp_shared001_min",
    #     "family": "RealMLP",
    #     "oof": "oof_exp_20260503_007_realmlp_shared001_min.npy",
    #     "pred": "pred_exp_20260503_007_realmlp_shared001_min.npy",
    #     "public_lb": 0.95273,
    # },
    # {
    #     "key": "009",
    #     "name": "009_lgb_shared003_full_fe_single",
    #     "family": "LightGBM",
    #     "oof": "oof_exp_20260503_009_lgb_shared003_full_fe_single_min.npy",
    #     "pred": "pred_exp_20260503_009_lgb_shared003_full_fe_single_min.npy",
    #     "public_lb": 0.95076,
    # },
    {
        "key": "014",
        "name": "cat_shared004_no_tyreageratio_str_min",
        "family": "CatBoost",
        "oof": "oof_exp_20260508_014_cat_shared004_no_tyreageratio_str_min.npy",
        "pred": "pred_exp_20260508_014_cat_shared004_no_tyreageratio_str_min.npy",
        "public_lb": 0.95233,
    },
    {
        "key": "015",
        "name": "cat_shared004_no_race_year_groupstats_min",
        "family": "CatBoost",
        "oof": "oof_exp_20260508_015_cat_shared004_no_race_year_groupstats_min.npy",
        "pred": "pred_exp_20260508_015_cat_shared004_no_race_year_groupstats_min.npy",
        "public_lb": 0.95244,
    },
    {
        "key": "016",
        "name": "xgb_shared005_fe_te_seedens_min",
        "family": "XGBoost",
        "oof": "oof_exp_20260508_016_xgb_shared005_fe_te_seedens_min.npy",
        "pred": "pred_exp_20260508_016_xgb_shared005_fe_te_seedens_min.npy",
        "public_lb": 0.95164,
    },
    {
        "key": "017",
        "name": "xgb_shared005_no_pitstop_pairte_min",
        "family": "XGBoost",
        "oof": "oof_exp_20260509_017_xgb_shared005_no_pitstop_pairte_min.npy",
        "pred": "pred_exp_20260509_017_xgb_shared005_no_pitstop_pairte_min.npy",
        "public_lb": 0.95164,
    },
    {
        "key": "018",
        "name": "realmlp_shared006_pipeline_a_min",
        "family": "RealMLP",
        "oof": "oof_exp_20260510_018_realmlp_shared006_pipeline_a_min.npy",
        "pred": "pred_exp_20260510_018_realmlp_shared006_pipeline_a_min.npy",
        "public_lb": 0.95316,
    },
    # {
    #     "key": "020",
    #     "name": "lgb_progress_window_te_min",
    #     "family": "LightGBM",
    #     "oof": "oof_exp_20260511_020_lgb_progress_window_te_min.npy",
    #     "pred": "pred_exp_20260511_020_lgb_progress_window_te_min.npy",
    #     "public_lb": 0.95075,
    # },
    {
        "key": "021",
        "name": "tabm_shared007_comp_oof_no_pitstop_no_isoriginaldata_min",
        "family": "TabM",
        "oof": "oof_exp_20260511_021_tabm_shared007_comp_oof_no_pitstop_no_isoriginaldata_min.npy",
        "pred": "pred_exp_20260511_021_tabm_shared007_comp_oof_no_pitstop_no_isoriginaldata_min.npy",
        "public_lb": 0.95171,
    },
    # {
    #     "key": "022",
    #     "name": "realmlp_shared006_pipeline_a_no_race_year_te_min",
    #     "family": "RealMLP",
    #     "oof": "oof_exp_20260512_022_realmlp_shared006_pipeline_a_no_race_year_te_min.npy",
    #     "pred": "pred_exp_20260512_022_realmlp_shared006_pipeline_a_no_race_year_te_min.npy",
    #     "public_lb": 0.95301,
    # },
    # {
    #     "key": "023",
    #     "name": "realmlp_shared006_pipeline_a_no_pitstop_cat_count_min",
    #     "family": "RealMLP",
    #     "oof": "oof_exp_20260512_023_realmlp_shared006_pipeline_a_no_pitstop_cat_count_min.npy",
    #     "pred": "pred_exp_20260512_023_realmlp_shared006_pipeline_a_no_pitstop_cat_count_min.npy",
    #     "public_lb": 0.95304,
    # },
    # {
    #     "key": "024",
    #     "name": "lgb_progress_window_weather_agg_min",
    #     "family": "LightGBM",
    #     "oof": "oof_exp_20260513_024_lgb_progress_window_weather_agg_min.npy",
    #     "pred": "pred_exp_20260513_024_lgb_progress_window_weather_agg_min.npy",
    #     "public_lb": 0.95086,
    # },
    # {
    #     "key": "025",
    #     "name": "lgb_year_stint_flags_min",
    #     "family": "LightGBM",
    #     "oof": "oof_exp_20260513_025_lgb_year_stint_flags_min.npy",
    #     "pred": "pred_exp_20260513_025_lgb_year_stint_flags_min.npy",
    #     "public_lb": 0.95092,
    # },
    {
        "key": "026",
        "name": "lgb_weather_year_stint_flags_min",
        "family": "LightGBM",
        "oof": "oof_exp_20260514_026_lgb_weather_year_stint_flags_min.npy",
        "pred": "pred_exp_20260514_026_lgb_weather_year_stint_flags_min.npy",
        "public_lb": 0.95096,
    },
    {
        "key": "027",
        "name": "realmlp_shared009_min",
        "family": "RealMLP",
        "oof": "oof_exp_20260514_027_realmlp_shared009_min.npy",
        "pred": "pred_exp_20260514_027_realmlp_shared009_min.npy",
        "public_lb": 0.95260,
    },
    {
        "key": "028",
        "name": "parent_child_mlp_shared010_min",
        "family": "Parent-Child MLP",
        "oof": "oof_exp_20260514_028_parent_child_mlp_shared010_min_all_models_mean.npy",
        "pred": "pred_exp_20260514_028_parent_child_mlp_shared010_min_all_models_mean.npy",
        "public_lb": 0.95260,
    },
    {
        "key": "029",
        "name": "realmlp_shared001v2_min",
        "family": "RealMLP",
        "oof": "oof_exp_20260514_029_realmlp_shared001v2_min.npy",
        "pred": "pred_exp_20260514_029_realmlp_shared001v2_min.npy",
        "public_lb": 0.95397,
    },
    {
        "key": "030",
        "name": "catboost_meta_stacking_shared011_min",
        "family": "CatBoost",
        "oof": "oof_exp_20260514_030_catboost_meta_stacking_shared011_min.npy",
        "pred": "pred_exp_20260514_030_catboost_meta_stacking_shared011_min.npy",
        "public_lb": 0.95214,
    },
    {
        "key": "031",
        "name": "catboost_meta_stacking_shared011_tawara_gpu_min",
        "family": "CatBoost",
        "oof": "oof_exp_20260514_031_catboost_meta_stacking_shared011_tawara_gpu_min.npy",
        "pred": "pred_exp_20260514_031_catboost_meta_stacking_shared011_tawara_gpu_min.npy",
        "public_lb": 0.95220,
    },
    {
        "key": "032",
        "name": "custom_realmlp_cv095352_min",
        "family": "RealMLP",
        "oof": "oof_exp_20260516_032_custom_realmlp_cv095352_min.npy",
        "pred": "pred_exp_20260516_032_custom_realmlp_cv095352_min.npy",
        "public_lb": 0.95309,
    },
]

oofs = {}
preds = {}

for spec in MODEL_SPECS:
    oof_path = CFG.NPY_BASE / spec["oof"]
    pred_path = CFG.NPY_BASE / spec["pred"]

    assert oof_path.exists(), f"missing oof: {oof_path}"
    assert pred_path.exists(), f"missing pred: {pred_path}"

    oof = np.load(oof_path)
    pred = np.load(pred_path)

    assert len(oof) == len(train), (spec["key"], len(oof), len(train))
    assert len(pred) == len(sub), (spec["key"], len(pred), len(sub))
    assert np.isfinite(oof).all(), spec["key"]
    assert np.isfinite(pred).all(), spec["key"]

    oofs[spec["key"]] = oof.astype(float)
    preds[spec["key"]] = pred.astype(float)

model_keys = [s["key"] for s in MODEL_SPECS]
model_names = [s["name"] for s in MODEL_SPECS]

X_raw = np.column_stack([oofs[k] for k in model_keys])
T_raw = np.column_stack([preds[k] for k in model_keys])

print("X_raw:", X_raw.shape)
print("T_raw:", T_raw.shape)

X_raw: (439140, 13)
T_raw: (188165, 13)


In [7]:
# ============================================================
# Individual reports
# ============================================================

individual_rows = []

for i, spec in enumerate(MODEL_SPECS):
    pred_oof = X_raw[:, i]
    individual_rows.append({
        "key": spec["key"],
        "name": spec["name"],
        "family": spec["family"],
        "cv_auc": auc(y, pred_oof),
        "public_lb": spec["public_lb"],
        "oof_min": float(pred_oof.min()),
        "oof_max": float(pred_oof.max()),
        "pred_min": float(T_raw[:, i].min()),
        "pred_max": float(T_raw[:, i].max()),
    })

individual_df = pd.DataFrame(individual_rows).sort_values("cv_auc", ascending=False)
display(individual_df)

best_single_key = individual_df.iloc[0]["key"]
best_single_auc = individual_df.iloc[0]["cv_auc"]

print("best single:", best_single_key, best_single_auc)

,key,name,family,cv_auc,public_lb,oof_min,oof_max,pred_min,pred_max
9,029,realmlp_shared001v2_min,RealMLP,0.954042,0.95397,4.969082e-08,0.998981,0.000008,0.998439
12,032,custom_realmlp_cv095352_min,RealMLP,0.953506,0.95309,3.416236e-04,0.998800,0.000474,0.998323
4,018,realmlp_shared006_pipeline_a_min,RealMLP,0.953476,0.95316,1.100145e-05,0.998269,0.000023,0.997298
7,027,realmlp_shared009_min,RealMLP,0.953140,0.95260,1.388308e-05,0.999154,0.000019,0.996669
11,031,catboost_meta_stacking_shared011_tawara_gpu_min,CatBoost,0.953093,0.95220,1.682323e-03,0.972204,0.001682,0.972037
10,030,catboost_meta_stacking_shared011_min,CatBoost,0.952983,0.95214,1.548887e-03,0.968364,0.001549,0.968364
0,014,cat_shared004_no_tyreageratio_str_min,CatBoost,0.952726,0.95233,7.059056e-05,0.997215,0.000153,0.996355
1,015,cat_shared004_no_race_year_groupstats_min,CatBoost,0.952714,0.95244,4.489366e-05,0.997058,0.000141,0.996636
2,016,xgb_shared005_fe_te_seedens_min,XGBoost,0.951986,0.95164,1.674493e-05,0.997698,0.000027,0.997298
3,017,xgb_shared005_no_pitstop_pairte_min,XGBoost,0.951939,0.95164,1.331864e-05,0.997702,0.000024,0.997621


best single: 029 0.9540420784720796


In [8]:
# ============================================================
# Correlation matrix
# ============================================================

pearson = pd.DataFrame(
    np.corrcoef(X_raw.T),
    index=model_keys,
    columns=model_keys,
)

spearman_mat = np.zeros((len(model_keys), len(model_keys)))

for i in range(len(model_keys)):
    for j in range(len(model_keys)):
        spearman_mat[i, j] = spearmanr(X_raw[:, i], X_raw[:, j]).correlation

spearman_df = pd.DataFrame(
    spearman_mat,
    index=model_keys,
    columns=model_keys,
)

print("Pearson correlation")
display(pearson)

print("Spearman correlation")
display(spearman_df)

pearson.to_csv(CFG.OUTDIR / "corr_pearson.csv")
spearman_df.to_csv(CFG.OUTDIR / "corr_spearman.csv")

Pearson correlation


,014,015,016,017,018,021,026,027,028,029,030,031,032
014,1.000000,0.996917,0.962898,0.962739,0.962634,0.968765,0.984963,0.966718,0.956846,0.961507,0.972616,0.972536,0.960927
015,0.996917,1.000000,0.962398,0.962237,0.961623,0.967914,0.984951,0.965957,0.956020,0.960397,0.971936,0.971857,0.959915
016,0.962898,0.962398,1.000000,0.998485,0.981123,0.978284,0.960326,0.983108,0.972366,0.980101,0.984009,0.983998,0.979115
017,0.962739,0.962237,0.998485,1.000000,0.980833,0.978150,0.960937,0.982845,0.972213,0.979812,0.984012,0.983990,0.979023
018,0.962634,0.961623,0.981123,0.980833,1.000000,0.982275,0.958341,0.992382,0.981724,0.994590,0.984836,0.984807,0.989928
021,0.968765,0.967914,0.978284,0.978150,0.982275,1.000000,0.966370,0.985771,0.974491,0.980413,0.984509,0.984433,0.982081
026,0.984963,0.984951,0.960326,0.960937,0.958341,0.966370,1.000000,0.963882,0.954081,0.957061,0.966154,0.966107,0.958169
027,0.966718,0.965957,0.983108,0.982845,0.992382,0.985771,0.963882,1.000000,0.980820,0.990541,0.987809,0.987754,0.989447
028,0.956846,0.956020,0.972366,0.972213,0.981724,0.974491,0.954081,0.980820,1.000000,0.981320,0.976647,0.976681,0.979124
029,0.961507,0.960397,0.980101,0.979812,0.994590,0.980413,0.957061,0.990541,0.981320,1.000000,0.983610,0.983652,0.990775


Spearman correlation


,014,015,016,017,018,021,026,027,028,029,030,031,032
014,1.000000,0.994676,0.975612,0.975146,0.976431,0.970041,0.973115,0.976887,0.968397,0.976608,0.987100,0.988247,0.972390
015,0.994676,1.000000,0.975249,0.974943,0.976373,0.969630,0.972735,0.977016,0.968181,0.976407,0.987522,0.988604,0.972607
016,0.975612,0.975249,1.000000,0.998203,0.973029,0.965983,0.969232,0.973150,0.967417,0.972249,0.973322,0.974503,0.966370
017,0.975146,0.974943,0.998203,1.000000,0.972656,0.965901,0.969808,0.972830,0.967124,0.972103,0.972933,0.974139,0.966202
018,0.976431,0.976373,0.973029,0.972656,1.000000,0.970461,0.966326,0.987949,0.975111,0.990889,0.972732,0.973860,0.979627
021,0.970041,0.969630,0.965983,0.965901,0.970461,1.000000,0.962785,0.972047,0.967054,0.967894,0.968747,0.969906,0.966383
026,0.973115,0.972735,0.969232,0.969808,0.966326,0.962785,1.000000,0.968291,0.959946,0.966439,0.972236,0.973791,0.965142
027,0.976887,0.977016,0.973150,0.972830,0.987949,0.972047,0.968291,1.000000,0.973925,0.985736,0.974447,0.975597,0.980497
028,0.968397,0.968181,0.967417,0.967124,0.975111,0.967054,0.959946,0.973925,1.000000,0.973048,0.967800,0.968876,0.967367
029,0.976608,0.976407,0.972249,0.972103,0.990889,0.967894,0.966439,0.985736,0.973048,1.000000,0.971664,0.972860,0.980598


In [9]:
# ============================================================
# Meta feature builders
# ============================================================

def build_meta_features(X, T, keys):
    X = np.asarray(X, dtype=float)
    T = np.asarray(T, dtype=float)

    X_rank = np.column_stack([rank01(X[:, i]) for i in range(X.shape[1])])
    T_rank = np.column_stack([rank01(T[:, i]) for i in range(T.shape[1])])

    X_logit = np.column_stack([safe_logit(X[:, i]) for i in range(X.shape[1])])
    T_logit = np.column_stack([safe_logit(T[:, i]) for i in range(T.shape[1])])

    X_meta = np.column_stack([X, X_rank, X_logit])
    T_meta = np.column_stack([T, T_rank, T_logit])

    feature_names = (
        [f"raw_{k}" for k in keys] +
        [f"rank_{k}" for k in keys] +
        [f"logit_{k}" for k in keys]
    )

    return X_meta, T_meta, feature_names


X_meta_full, T_meta_full, meta_feature_names_full = build_meta_features(
    X_raw,
    T_raw,
    model_keys,
)

print("X_meta_full:", X_meta_full.shape)
print("T_meta_full:", T_meta_full.shape)
print("meta meta_feature_names_full:", meta_feature_names_full)

X_meta_full: (439140, 39)
T_meta_full: (188165, 39)
meta meta_feature_names_full: ['raw_014', 'raw_015', 'raw_016', 'raw_017', 'raw_018', 'raw_021', 'raw_026', 'raw_027', 'raw_028', 'raw_029', 'raw_030', 'raw_031', 'raw_032', 'rank_014', 'rank_015', 'rank_016', 'rank_017', 'rank_018', 'rank_021', 'rank_026', 'rank_027', 'rank_028', 'rank_029', 'rank_030', 'rank_031', 'rank_032', 'logit_014', 'logit_015', 'logit_016', 'logit_017', 'logit_018', 'logit_021', 'logit_026', 'logit_027', 'logit_028', 'logit_029', 'logit_030', 'logit_031', 'logit_032']


In [10]:
# ============================================================
# Save candidate helper
# ============================================================

candidate_records = {}
candidate_summary = []

def add_candidate(
    name,
    method,
    oof_pred,
    test_pred,
    weights=None,
    params=None,
    notes=None,
):
    oof_pred = np.asarray(oof_pred, dtype=float)
    test_pred = np.asarray(test_pred, dtype=float)

    score = auc(y, oof_pred)

    candidate_records[name] = {
        "method": method,
        "oof": oof_pred,
        "pred": test_pred,
        "score": float(score),
        "weights": None if weights is None else [float(x) for x in weights],
        "params": params or {},
        "notes": notes or "",
    }

    candidate_summary.append({
        "name": name,
        "method": method,
        "cv_auc": float(score),
        f"delta_vs_{best_single_key}": float(score - auc(y, oofs[best_single_key])),
        "delta_vs_best_single": float(score - best_single_auc),
        "weights": None if weights is None else json.dumps([round(float(x), 8) for x in weights]),
        "params": json.dumps(params or {}, ensure_ascii=False),
        "notes": notes or "",
        "oof_min": float(oof_pred.min()),
        "oof_max": float(oof_pred.max()),
        "pred_min": float(test_pred.min()),
        "pred_max": float(test_pred.max()),
    })

    print(f"{name}: {score:.9f}")

In [11]:
# ============================================================
# Simple averages
# ============================================================

print("\n" + "=" * 80)
print("Simple averages")
print("=" * 80)

avg_specs = {
    # "avg_007_014_015_016_018_020_021_022_023": ["007", "014", "015", "016", "018", "020", "021", "022", "023"],
    # "avg_007_014_016_018_020_021_022_023": ["007", "014", "016", "018", "020", "021", "022", "023"],
    # "avg_007_014_015_018_020_021_022_023": ["007", "014", "015", "018", "020", "021", "022", "023"],
    "avg" : model_keys
}

for name, keys in avg_specs.items():
    idx = [model_keys.index(k) for k in keys]
    w = np.zeros(len(model_keys))
    for j in idx:
        w[j] = 1.0 / len(idx)

    add_candidate(
        name=name,
        method="avg",
        oof_pred=weighted_average(X_raw, w),
        test_pred=weighted_average(T_raw, w),
        weights=w,
        notes=f"simple average of {keys}",
    )


Simple averages
avg: 0.954554429


In [12]:
# ============================================================
# Hill Climb non-negative weights
# ============================================================

print("\n" + "=" * 80)
print("Hill Climb")
print("=" * 80)

def hill_climb_weights(X, y, init_candidates=None):
    n = X.shape[1]

    candidates = []

    # one-hot starts
    for i in range(n):
        w = np.zeros(n)
        w[i] = 1.0
        candidates.append(w)

    # avg start
    candidates.append(np.ones(n) / n)

    if init_candidates:
        for w in init_candidates:
            candidates.append(normalize_weights(w))

    best_w = None
    best_score = -np.inf

    for w in candidates:
        s = auc(y, weighted_average(X, w))
        if s > best_score:
            best_score = s
            best_w = normalize_weights(w)

    for step in [0.20, 0.10, 0.05, 0.02, 0.01, 0.005, 0.002, 0.001]:
        improved = True

        while improved:
            improved = False

            for i in range(n):
                for direction in [-1, 1]:
                    trial = best_w.copy()
                    trial[i] += direction * step
                    trial = normalize_weights(trial)

                    s = auc(y, weighted_average(X, trial))

                    if s > best_score + 1e-12:
                        best_score = s
                        best_w = trial
                        improved = True

    return best_w, best_score


hc_init = []

# Use candidates that already looked good as starts
for cand_name in ["avg_007_008_009", "avg_006b_007_008_009"]:
    if cand_name in candidate_records:
        hc_init.append(candidate_records[cand_name]["weights"])

hc_w, hc_score = hill_climb_weights(X_raw, y, init_candidates=hc_init)

add_candidate(
    name="hc_nonnegative_raw",
    method="hc",
    oof_pred=weighted_average(X_raw, hc_w),
    test_pred=weighted_average(T_raw, hc_w),
    weights=hc_w,
    params={"constraint": "nonnegative_sum1"},
    notes="greedy hill climb on raw OOF predictions; in-sample optimizer, interpret cautiously",
)

print("HC weights:")
for k, w in zip(model_keys, hc_w):
    print(k, w)


Hill Climb
hc_nonnegative_raw: 0.954847796
HC weights:
014 0.010749140480032457
015 0.01093662662764746
016 0.10641560937765078
017 0.007997121730184995
018 0.0
021 0.018835184590323326
026 0.039717582327751105
027 0.0
028 0.06105214104700602
029 0.4071581988139831
030 0.0
031 0.17745518138543726
032 0.15968321361998367


In [13]:
# ============================================================
# Nelder-Mead softmax weights
# ============================================================

print("\n" + "=" * 80)
print("Nelder-Mead")
print("=" * 80)

def nm_optimize_weights(X, y, start_w=None, maxiter=500):
    n = X.shape[1]

    if start_w is None:
        start_w = np.ones(n) / n

    start_w = normalize_weights(start_w)
    z0 = np.log(np.clip(start_w, 1e-8, 1.0))

    def objective(z):
        w = softmax(z)
        p = weighted_average(X, w)
        return -auc(y, p)

    res = minimize(
        objective,
        z0,
        method="Nelder-Mead",
        options={
            "maxiter": maxiter,
            "xatol": 1e-7,
            "fatol": 1e-10,
            "disp": True,
        },
    )

    w = softmax(res.x)
    score = auc(y, weighted_average(X, w))

    return w, score, res


nm_w, nm_score, nm_res = nm_optimize_weights(
    X_raw,
    y,
    start_w=hc_w,
    maxiter=500,
)

add_candidate(
    name="nm_softmax_raw",
    method="nm",
    oof_pred=weighted_average(X_raw, nm_w),
    test_pred=weighted_average(T_raw, nm_w),
    weights=nm_w,
    params={
        "constraint": "softmax_sum1",
        "success": bool(nm_res.success),
        "message": str(nm_res.message),
        "nit": int(getattr(nm_res, "nit", -1)),
        "fun": float(nm_res.fun),
    },
    notes="Nelder-Mead on softmax weights; in-sample optimizer, interpret cautiously",
)

print("NM weights:")
for k, w in zip(model_keys, nm_w):
    print(k, w)


Nelder-Mead
Optimization terminated successfully.
         Current function value: -0.954848
         Iterations: 211
         Function evaluations: 589
nm_softmax_raw: 0.954847814
NM weights:
014 0.0108168591476576
015 0.010858949348742604
016 0.10624161700363445
017 0.007952481344483578
018 9.542849118237539e-09
021 0.017706111015129544
026 0.03951032975099455
027 9.619554311338707e-09
028 0.06122648762532239
029 0.40807402768712847
030 9.616727377967949e-09
031 0.1777148688877383
032 0.1598982394100376


In [14]:
# ============================================================
# Signed SLSQP weights
# Allow small negative weights, with sum(w)=1
# ============================================================

def weighted_average_signed(X, w):
    return np.asarray(X, dtype=float) @ np.asarray(w, dtype=float)


def optimize_signed_slsqp(
    X,
    y,
    start_w=None,
    neg_bound=-0.10,
    pos_bound=0.60,
    maxiter=1000,
):
    n = X.shape[1]

    if start_w is None:
        start_w = np.ones(n) / n
    else:
        start_w = np.asarray(start_w, dtype=float)

    # Start must satisfy sum=1
    start_w = start_w / start_w.sum()

    bounds = [(neg_bound, pos_bound) for _ in range(n)]

    constraints = [
        {
            "type": "eq",
            "fun": lambda w: np.sum(w) - 1.0,
        }
    ]

    def objective(w):
        p = weighted_average_signed(X, w)
        return -auc(y, p)

    res = minimize(
        objective,
        start_w,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={
            "maxiter": maxiter,
            "ftol": 1e-12,
            "disp": True,
        },
    )

    w = res.x
    score = auc(y, weighted_average_signed(X, w))

    return w, score, res

In [15]:
print("\n" + "=" * 80)
print("Signed SLSQP")
print("=" * 80)

signed_w, signed_score, signed_res = optimize_signed_slsqp(
    X_raw,
    y,
    start_w=nm_w,
    neg_bound=-0.05,
    pos_bound=0.50,
    maxiter=1000,
)

add_candidate(
    name="slsqp_signed_raw_neg005",
    method="slsqp_signed",
    oof_pred=weighted_average_signed(X_raw, signed_w),
    test_pred=weighted_average_signed(T_raw, signed_w),
    weights=signed_w,
    params={
        "constraint": "sum1_signed",
        "neg_bound": -0.05,
        "pos_bound": 0.50,
        "success": bool(signed_res.success),
        "message": str(signed_res.message),
        "nit": int(getattr(signed_res, "nit", -1)),
        "fun": float(signed_res.fun),
    },
    notes="signed SLSQP on raw OOF predictions; high overfit risk; attack-only",
)

print("Signed SLSQP weights:")
for k, w in zip(model_keys, signed_w):
    print(k, w)

print("Signed score:", signed_score)
print("sum weights:", signed_w.sum())
print("min weight:", signed_w.min())
print("max weight:", signed_w.max())


Signed SLSQP
Optimization terminated successfully    (Exit mode 0)
            Current function value: -0.9548477944870721
            Iterations: 18
            Function evaluations: 430
            Gradient evaluations: 18
slsqp_signed_raw_neg005: 0.954847794
Signed SLSQP weights:
014 0.01070474544627885
015 0.01072114166211604
016 0.10622134155027815
017 0.007753727183149881
018 0.0002242091966023158
021 0.017628173478083178
026 0.03962430377747693
027 -0.00018903602288985254
028 0.06115515328062225
029 0.4082484803005699
030 -1.9742810813878225e-05
031 0.1779751547560344
032 0.15995234820249157
Signed score: 0.9548477944870721
sum weights: 0.9999999999999999
min weight: -0.00018903602288985254
max weight: 0.4082484803005699


In [16]:
# ============================================================
# Prune models based on HC / NM / SLSQP weights
# ============================================================

PRUNE_HARD_THRESHOLD = 0.005
PRUNE_SOFT_THRESHOLD = 0.020
KEEP_PROTECTED_KEYS = {
    "007",  # RealMLP old core
    "014", "015",  # CatBoost branch
    "016", "017",  # XGB branch; 017 is historically useful for LogReg
    "018", "022", "023",  # RealMLP shared006 branch
    "021",  # TabM branch
}

def build_pruned_model_keys(model_keys, hc_w, nm_w, signed_w):
    rows = []
    for k, wh, wn, ws in zip(model_keys, hc_w, nm_w, signed_w):
        max_w = max(float(wh), float(wn), float(ws))
        min_w = min(float(wh), float(wn), float(ws))

        if max_w < PRUNE_HARD_THRESHOLD and k not in KEEP_PROTECTED_KEYS:
            decision = "drop_hard"
        elif max_w < PRUNE_SOFT_THRESHOLD and k not in KEEP_PROTECTED_KEYS:
            decision = "drop_soft"
        else:
            decision = "keep"

        rows.append({
            "key": k,
            "hc_weight": float(wh),
            "nm_weight": float(wn),
            "slsqp_weight": float(ws),
            "max_weight": max_w,
            "min_weight": min_w,
            "protected": k in KEEP_PROTECTED_KEYS,
            "decision": decision,
        })

    prune_df = pd.DataFrame(rows)
    keep_keys = prune_df.loc[prune_df["decision"] == "keep", "key"].tolist()
    drop_keys = prune_df.loc[prune_df["decision"] != "keep", "key"].tolist()

    return keep_keys, drop_keys, prune_df


keep_keys, drop_keys, prune_df = build_pruned_model_keys(
    model_keys=model_keys,
    hc_w=hc_w,
    nm_w=nm_w,
    signed_w=signed_w,
)

print("keep_keys:", keep_keys)
print("drop_keys:", drop_keys)
display(prune_df)

prune_df.to_csv(CFG.OUTDIR / "prune_decision_after_hc_nm_slsqp.csv", index=False)

keep_keys: ['014', '015', '016', '017', '018', '021', '026', '028', '029', '031', '032']
drop_keys: ['027', '030']


,key,hc_weight,nm_weight,slsqp_weight,max_weight,min_weight,protected,decision
0,014,0.010749,1.081686e-02,0.010705,1.081686e-02,0.010705,True,keep
1,015,0.010937,1.085895e-02,0.010721,1.093663e-02,0.010721,True,keep
2,016,0.106416,1.062416e-01,0.106221,1.064156e-01,0.106221,True,keep
3,017,0.007997,7.952481e-03,0.007754,7.997122e-03,0.007754,True,keep
4,018,0.000000,9.542849e-09,0.000224,2.242092e-04,0.000000,True,keep
5,021,0.018835,1.770611e-02,0.017628,1.883518e-02,0.017628,True,keep
6,026,0.039718,3.951033e-02,0.039624,3.971758e-02,0.039510,False,keep
7,027,0.000000,9.619554e-09,-0.000189,9.619554e-09,-0.000189,False,drop_hard
8,028,0.061052,6.122649e-02,0.061155,6.122649e-02,0.061052,False,keep
9,029,0.407158,4.080740e-01,0.408248,4.082485e-01,0.407158,False,keep


In [17]:
# ============================================================
# Rebuild matrices for pruned Ridge / LogReg
# ============================================================

pruned_idx = [model_keys.index(k) for k in keep_keys]

model_keys_full = model_keys.copy()
model_keys_pruned = keep_keys.copy()

X_raw_full = X_raw
T_raw_full = T_raw

X_raw_pruned = X_raw[:, pruned_idx]
T_raw_pruned = T_raw[:, pruned_idx]

# Temporarily switch global model_keys for meta feature names
model_keys = model_keys_pruned
X_meta, T_meta, meta_feature_names = build_meta_features(
    X_raw_pruned,
    T_raw_pruned,
    keep_keys,
)

print("Pruned X_raw:", X_raw_pruned.shape)
print("Pruned X_meta:", X_meta.shape)
print("Pruned model_keys:", model_keys)

Pruned X_raw: (439140, 11)
Pruned X_meta: (439140, 33)
Pruned model_keys: ['014', '015', '016', '017', '018', '021', '026', '028', '029', '031', '032']


In [18]:
# ============================================================
# Ridge / ElasticNet / LogisticRegression meta CV
# Two-stage search
# ============================================================

meta_folds = list(
    StratifiedKFold(
        n_splits=CFG.N_SPLITS,
        shuffle=True,
        random_state=CFG.SEED,
    ).split(X_meta, y)
)


def make_refined_grid_1d(best_value, factor_low=0.25, factor_high=4.0, n=17, min_value=1e-8):
    lo = max(best_value * factor_low, min_value)
    hi = max(best_value * factor_high, lo * 1.01)
    return np.geomspace(lo, hi, n).tolist()


def run_meta_regressor_cv(estimator_factory, params):
    oof_meta = np.zeros(len(y), dtype=float)

    for tr_idx, va_idx in meta_folds:
        model = make_pipeline(
            StandardScaler(),
            estimator_factory(params),
        )
        model.fit(X_meta[tr_idx], y[tr_idx])
        oof_meta[va_idx] = model.predict(X_meta[va_idx])

    score = auc(y, oof_meta)
    return score, oof_meta


def run_meta_regressor_two_stage(name, estimator_factory, coarse_grid, refine_builder):
    history = []

    best = None

    print(f"\n{name} coarse search")
    for params in coarse_grid:
        score, oof_meta = run_meta_regressor_cv(estimator_factory, params)
        history.append({"stage": "coarse", "score": float(score), **params})
        print(params, score)

        if best is None or score > best["score"]:
            best = {
                "score": score,
                "params": params,
                "oof": oof_meta,
            }

    refined_grid = refine_builder(best["params"])

    print(f"\n{name} refined search around {best['params']}")
    for params in refined_grid:
        score, oof_meta = run_meta_regressor_cv(estimator_factory, params)
        history.append({"stage": "refined", "score": float(score), **params})
        print(params, score)

        if score > best["score"]:
            best = {
                "score": score,
                "params": params,
                "oof": oof_meta,
            }

    final_model = make_pipeline(
        StandardScaler(),
        estimator_factory(best["params"]),
    )
    final_model.fit(X_meta, y)
    test_meta = final_model.predict(T_meta)

    add_candidate(
        name=name,
        method=name.split("_")[0],
        oof_pred=best["oof"],
        test_pred=test_meta,
        params=best["params"],
        notes="two-stage meta CV on raw+rank+logit features",
    )

    hist_df = pd.DataFrame(history).sort_values("score", ascending=False)
    hist_df.to_csv(CFG.OUTDIR / f"{name}_search_history.csv", index=False)

    print(f"\n{name} best:", best["params"], best["score"])
    display(hist_df.head(30))

    return best, hist_df


def run_meta_logreg_cv(params):
    oof_meta = np.zeros(len(y), dtype=float)

    for tr_idx, va_idx in meta_folds:
        model = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                C=params["C"],
                penalty="l2",
                solver="lbfgs",
                max_iter=3000,
                random_state=CFG.SEED,
            ),
        )
        model.fit(X_meta[tr_idx], y[tr_idx])
        oof_meta[va_idx] = model.predict_proba(X_meta[va_idx])[:, 1]

    score = auc(y, oof_meta)
    return score, oof_meta


def run_meta_logreg_two_stage(name, coarse_grid, refine_builder):
    history = []
    best = None

    print(f"\n{name} coarse search")
    for params in coarse_grid:
        score, oof_meta = run_meta_logreg_cv(params)
        history.append({"stage": "coarse", "score": float(score), **params})
        print(params, score)

        if best is None or score > best["score"]:
            best = {
                "score": score,
                "params": params,
                "oof": oof_meta,
            }

    refined_grid = refine_builder(best["params"])

    print(f"\n{name} refined search around {best['params']}")
    for params in refined_grid:
        score, oof_meta = run_meta_logreg_cv(params)
        history.append({"stage": "refined", "score": float(score), **params})
        print(params, score)

        if score > best["score"]:
            best = {
                "score": score,
                "params": params,
                "oof": oof_meta,
            }

    final_model = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=best["params"]["C"],
            penalty="l2",
            solver="lbfgs",
            max_iter=3000,
            random_state=CFG.SEED,
        ),
    )
    final_model.fit(X_meta, y)
    test_meta = final_model.predict_proba(T_meta)[:, 1]

    add_candidate(
        name=name,
        method="logreg",
        oof_pred=best["oof"],
        test_pred=test_meta,
        params=best["params"],
        notes="two-stage meta CV logistic regression on raw+rank+logit features",
    )

    hist_df = pd.DataFrame(history).sort_values("score", ascending=False)
    hist_df.to_csv(CFG.OUTDIR / f"{name}_search_history.csv", index=False)

    print(f"\n{name} best:", best["params"], best["score"])
    display(hist_df.head(30))

    return best, hist_df


print("\n" + "=" * 80)
print("Ridge / ElasticNet / LogReg two-stage search")
print("=" * 80)


# ----------------------------
# Ridge two-stage
# ----------------------------

ridge_coarse_grid = [
    {"alpha": a}
    for a in np.geomspace(1e-3, 1e3, 19)
]

ridge_best, ridge_hist = run_meta_regressor_two_stage(
    name="ridge_meta_pruned",
    estimator_factory=lambda p: Ridge(
        alpha=p["alpha"],
        random_state=CFG.SEED,
    ),
    coarse_grid=ridge_coarse_grid,
    refine_builder=lambda best_p: [
        {"alpha": a}
        for a in make_refined_grid_1d(
            best_p["alpha"],
            factor_low=0.2,
            factor_high=5.0,
            n=21,
            min_value=1e-6,
        )
    ],
)


# ----------------------------
# ElasticNet two-stage
# ----------------------------

elastic_coarse_grid = [
    {"alpha": a, "l1_ratio": l1}
    for a in np.geomspace(1e-4, 1e-1, 7)
    for l1 in [0.05, 0.10, 0.20, 0.50, 0.90]
]


def build_elastic_refined_grid_fast(best_p):
    alpha_grid = make_refined_grid_1d(
        best_p["alpha"],
        factor_low=0.5,
        factor_high=2.0,
        n=7,
        min_value=1e-7,
    )

    l1_center = best_p["l1_ratio"]
    l1_grid = sorted(set([
        round(max(0.001, l1_center - 0.10), 6),
        round(l1_center, 6),
        round(min(0.999, l1_center + 0.10), 6),
        0.05,
        0.10,
        0.20,
        0.50,
        0.90,
    ]))

    return [
        {"alpha": a, "l1_ratio": l1}
        for a in alpha_grid
        for l1 in l1_grid
    ]


# elastic_best, elastic_hist = run_meta_regressor_two_stage(
#     name="elasticnet_meta_all",
#     estimator_factory=lambda p: ElasticNet(
#         alpha=p["alpha"],
#         l1_ratio=p["l1_ratio"],
#         max_iter=30000,
#         random_state=CFG.SEED,
#         selection="cyclic",
#     ),
#     coarse_grid=elastic_coarse_grid,
#     refine_builder=build_elastic_refined_grid_fast,
# )


# ----------------------------
# LogisticRegression two-stage
# ----------------------------

logreg_coarse_grid = [
    {"C": c}
    for c in np.geomspace(1e-3, 1e3, 19)
]

logreg_best, logreg_hist = run_meta_logreg_two_stage(
    name="logreg_meta_pruned",
    coarse_grid=logreg_coarse_grid,
    refine_builder=lambda best_p: [
        {"C": c}
        for c in make_refined_grid_1d(
            best_p["C"],
            factor_low=0.2,
            factor_high=5.0,
            n=21,
            min_value=1e-6,
        )
    ],
)


Ridge / ElasticNet / LogReg two-stage search

ridge_meta_pruned coarse search
{'alpha': np.float64(0.001)} 0.9547183547851973
{'alpha': np.float64(0.0021544346900318843)} 0.9547183546225272
{'alpha': np.float64(0.004641588833612777)} 0.9547183576156573
{'alpha': np.float64(0.01)} 0.9547183583639399
{'alpha': np.float64(0.021544346900318832)} 0.9547183677337385
{'alpha': np.float64(0.046415888336127774)} 0.9547183908003609
{'alpha': np.float64(0.1)} 0.9547184366082656
{'alpha': np.float64(0.21544346900318823)} 0.9547185283542109
{'alpha': np.float64(0.46415888336127775)} 0.9547187352705984
{'alpha': np.float64(1.0)} 0.954719146793457
{'alpha': np.float64(2.154434690031882)} 0.954719915084414
{'alpha': np.float64(4.6415888336127775)} 0.9547213092323745
{'alpha': np.float64(10.0)} 0.9547234989022694
{'alpha': np.float64(21.54434690031882)} 0.9547264857858676
{'alpha': np.float64(46.41588833612773)} 0.954730556084971
{'alpha': np.float64(100.0)} 0.9547360669906202
{'alpha': np.float64(215

,stage,score,alpha
29,refined,0.954740,215.443469
16,coarse,0.954740,215.443469
30,refined,0.954740,253.063980
28,refined,0.954740,183.415626
31,refined,0.954740,297.253745
27,refined,0.954739,156.149045
32,refined,0.954738,349.159879
26,refined,0.954738,132.935916
25,refined,0.954737,113.173652
15,coarse,0.954736,100.000000



logreg_meta_pruned coarse search
{'C': np.float64(0.001)} 0.9547634864376013
{'C': np.float64(0.0021544346900318843)} 0.9547848638610055
{'C': np.float64(0.004641588833612777)} 0.9547961424633484
{'C': np.float64(0.01)} 0.9547997289164595
{'C': np.float64(0.021544346900318832)} 0.9548003897150039
{'C': np.float64(0.046415888336127774)} 0.9547978966653432
{'C': np.float64(0.1)} 0.9547981425900243
{'C': np.float64(0.21544346900318823)} 0.9547989824558321
{'C': np.float64(0.46415888336127775)} 0.9547988127583673
{'C': np.float64(1.0)} 0.9547993638847196
{'C': np.float64(2.154434690031882)} 0.9548040070429029
{'C': np.float64(4.6415888336127775)} 0.954797636718498
{'C': np.float64(10.0)} 0.9548042580428917
{'C': np.float64(21.54434690031882)} 0.9547999581837208
{'C': np.float64(46.41588833612773)} 0.9547987862756724
{'C': np.float64(100.0)} 0.9547991402783784
{'C': np.float64(215.44346900318823)} 0.9547991082323657
{'C': np.float64(464.1588833612773)} 0.9547979586101233
{'C': np.float64(1

,stage,score,C
29,refined,0.954804,10.000000
12,coarse,0.954804,10.000000
10,coarse,0.954804,2.154435
23,refined,0.954804,3.807308
20,refined,0.954803,2.349238
24,refined,0.954801,4.472136
22,refined,0.954800,3.241313
4,coarse,0.954800,0.021544
13,coarse,0.954800,21.544347
34,refined,0.954800,22.360680


In [19]:
# ============================================================
# Summary
# ============================================================

summary_df = pd.DataFrame(candidate_summary).sort_values("cv_auc", ascending=False)
display(summary_df)

summary_df.to_csv(CFG.OUTDIR / "blend_summary.csv", index=False)

# Expand weights table
weights_rows = []

for name, rec in candidate_records.items():
    if rec["weights"] is None:
        continue

    row = {
        "candidate": name,
        "method": rec["method"],
        "cv_auc": rec["score"],
    }

    for k, w in zip(model_keys, rec["weights"]):
        row[f"w_{k}"] = float(w)

    weights_rows.append(row)

weights_df = pd.DataFrame(weights_rows).sort_values("cv_auc", ascending=False)
display(weights_df)

weights_df.to_csv(CFG.OUTDIR / "blend_weights.csv", index=False)

,name,method,cv_auc,delta_vs_029,delta_vs_best_single,weights,params,notes,oof_min,oof_max,pred_min,pred_max
2,nm_softmax_raw,nm,0.954848,0.000806,0.000806,"[0.01081686, 0.01085895, 0.10624162, 0.0079524...","{""constraint"": ""softmax_sum1"", ""success"": true...",Nelder-Mead on softmax weights; in-sample opti...,0.000438,0.991813,0.000445,0.992593
1,hc_nonnegative_raw,hc,0.954848,0.000806,0.000806,"[0.01074914, 0.01093663, 0.10641561, 0.0079971...","{""constraint"": ""nonnegative_sum1""}",greedy hill climb on raw OOF predictions; in-s...,0.000438,0.991816,0.000445,0.992599
3,slsqp_signed_raw_neg005,slsqp_signed,0.954848,0.000806,0.000806,"[0.01070475, 0.01072114, 0.10622134, 0.0077537...","{""constraint"": ""sum1_signed"", ""neg_bound"": -0....",signed SLSQP on raw OOF predictions; high over...,0.000439,0.991808,0.000446,0.992589
5,logreg_meta_pruned,logreg,0.954804,0.000762,0.000762,None,"{""C"": 10.0}",two-stage meta CV logistic regression on raw+r...,0.000057,0.990285,0.000065,0.991492
4,ridge_meta_pruned,ridge,0.954740,0.000698,0.000698,None,"{""alpha"": 215.44346900318823}",two-stage meta CV on raw+rank+logit features,-0.015654,0.989970,-0.008545,0.987180
0,avg,avg,0.954554,0.000512,0.000512,"[0.07692308, 0.07692308, 0.07692308, 0.0769230...",{},"simple average of ['014', '015', '016', '017',...",0.000363,0.991526,0.000395,0.991973


,candidate,method,cv_auc,w_014,w_015,w_016,w_017,w_018,w_021,w_026,w_028,w_029,w_031,w_032
2,nm_softmax_raw,nm,0.954848,0.010817,0.010859,0.106242,0.007952,9.542849e-09,0.017706,0.039510,9.619554e-09,0.061226,0.408074,9.616727e-09
1,hc_nonnegative_raw,hc,0.954848,0.010749,0.010937,0.106416,0.007997,0.000000e+00,0.018835,0.039718,0.000000e+00,0.061052,0.407158,0.000000e+00
3,slsqp_signed_raw_neg005,slsqp_signed,0.954848,0.010705,0.010721,0.106221,0.007754,2.242092e-04,0.017628,0.039624,-1.890360e-04,0.061155,0.408248,-1.974281e-05
0,avg,avg,0.954554,0.076923,0.076923,0.076923,0.076923,7.692308e-02,0.076923,0.076923,7.692308e-02,0.076923,0.076923,7.692308e-02


In [20]:
# ============================================================
# Save OOF / pred / submissions
# ============================================================

print("\n" + "=" * 80)
print("Save blend artifacts")
print("=" * 80)

submission_dir = CFG.OUTDIR / "submissions"
submission_dir.mkdir(parents=True, exist_ok=True)

target_col = [c for c in sub.columns if c != CFG.ID_COL][0]

for name, rec in candidate_records.items():
    safe_name = sanitize_name(name)

    oof_path = CFG.OUTDIR / f"oof_{CFG.EXP_ID}_{safe_name}.npy"
    pred_path = CFG.OUTDIR / f"pred_{CFG.EXP_ID}_{safe_name}.npy"
    sub_path = submission_dir / f"sub_{safe_name}_{CFG.EXP_ID}.csv"

    np.save(oof_path, rec["oof"])
    np.save(pred_path, rec["pred"])

    sub_out = sub.copy()
    sub_out[target_col] = safe_clip(rec["pred"])
    sub_out.to_csv(sub_path, index=False)

    print(name, rec["score"], sub_path)


Save blend artifacts
avg 0.9545544291163748 /kaggle/working/exp_20260504_010_blend_core_min/submissions/sub_avg_exp_20260504_010_blend_core_min.csv
hc_nonnegative_raw 0.9548477957558991 /kaggle/working/exp_20260504_010_blend_core_min/submissions/sub_hc_nonnegative_raw_exp_20260504_010_blend_core_min.csv
nm_softmax_raw 0.95484781387735 /kaggle/working/exp_20260504_010_blend_core_min/submissions/sub_nm_softmax_raw_exp_20260504_010_blend_core_min.csv
slsqp_signed_raw_neg005 0.9548477944870721 /kaggle/working/exp_20260504_010_blend_core_min/submissions/sub_slsqp_signed_raw_neg005_exp_20260504_010_blend_core_min.csv
ridge_meta_pruned 0.9547404490958017 /kaggle/working/exp_20260504_010_blend_core_min/submissions/sub_ridge_meta_pruned_exp_20260504_010_blend_core_min.csv
logreg_meta_pruned 0.9548042580428917 /kaggle/working/exp_20260504_010_blend_core_min/submissions/sub_logreg_meta_pruned_exp_20260504_010_blend_core_min.csv


In [21]:
# ============================================================
# Save result json
# ============================================================

result = {
    "experiment": {
        "id": CFG.EXP_ID,
        "competition": CFG.COMPETITION,
        "metric": CFG.METRIC,
        "created_at": "2026-05-04",
    },
    "inputs": {
        "model_specs": MODEL_SPECS,
        "model_keys": model_keys,
        "n_models": len(model_keys),
    },
    "individual": individual_df.to_dict(orient="records"),
    "correlation": {
        "pearson": pearson.to_dict(),
        "spearman": spearman_df.to_dict(),
    },
    "blend_summary": summary_df.to_dict(orient="records"),
    "blend_weights": weights_df.to_dict(orient="records"),
    "best": {
        "candidate": str(summary_df.iloc[0]["name"]),
        "method": str(summary_df.iloc[0]["method"]),
        "cv_auc": float(summary_df.iloc[0]["cv_auc"]),
    },
    "notes": [
        "",
    ],
    "artifacts": {
        "outdir": str(CFG.OUTDIR),
        "blend_summary": str(CFG.OUTDIR / "blend_summary.csv"),
        "blend_weights": str(CFG.OUTDIR / "blend_weights.csv"),
        "submissions_dir": str(submission_dir),
    },
}

with open(CFG.OUTDIR / "blend_result.json", "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print("\nSaved to:", CFG.OUTDIR)
print("Best candidate:")
print(summary_df.iloc[0].to_dict())


Saved to: /kaggle/working/exp_20260504_010_blend_core_min
Best candidate:
{'name': 'nm_softmax_raw', 'method': 'nm', 'cv_auc': 0.95484781387735, 'delta_vs_029': 0.0008057354052704024, 'delta_vs_best_single': 0.0008057354052704024, 'weights': '[0.01081686, 0.01085895, 0.10624162, 0.00795248, 1e-08, 0.01770611, 0.03951033, 1e-08, 0.06122649, 0.40807403, 1e-08, 0.17771487, 0.15989824]', 'params': '{"constraint": "softmax_sum1", "success": true, "message": "Optimization terminated successfully.", "nit": 211, "fun": -0.95484781387735}', 'notes': 'Nelder-Mead on softmax weights; in-sample optimizer, interpret cautiously', 'oof_min': 0.0004382627378641478, 'oof_max': 0.9918128798032758, 'pred_min': 0.0004454711041903375, 'pred_max': 0.9925932006805667}
